## Test de verificación de funcionalidad correcta de las podas
A partir de un directorio de entrada, verifica si el número de vecinos para cada combinación única es el mismo usando los dos modos de reordenamiento y no usando ninguno.
Se realizaron las pruebas con los resultados de las búsquedas aleatorias sobre las 9 nubes de puntos con las que se trabajó y los resultados fueron de 100% de coincidencia.

In [1]:
import os
import glob
import pandas as pd
import numpy as np

BASE_DATA_PATH = os.path.join("..", "out_tfg_reorders_v3")
SUBSET_DATA = os.path.join(BASE_DATA_PATH, "subset")
ALL_REORDER_MODES = ["none", "polar", "cartesian"]
ALL_CLOUDS = ["bildstein_station1_xyz_intensity_rgb", "sg27_station8_intensity_rgb", "5085_54320", "5095_54440", "Lille_0", "Paris_Luxembourg_6", "PNOA_2024_PNR_489-4672_NPC01", 
              "Mar18_train", "Mar18_test"]

In [2]:
def verificar_integridad_vecinos(data_path, clouds):
    """
    Recorre los directorios de resultados de los datasets indicados, abre cada CSV
    y comprueba que el 'avg_result_size' sea idéntico para los modos de reordenación
    segmentando por el tipo de algoritmo ('neighborsPrune' y 'neighborsStruct').
    """
    total_errores = 0
    total_archivos_revisados = 0
    
    print("Iniciando test de validación de resultados estructurado por operaciones...")
    print("-" * 75)
    
    # 1. Iterar sobre los datasets proporcionados
    for dataset in clouds:
        dir_dataset = os.path.join(data_path, dataset)
        
        if not os.path.isdir(dir_dataset):
            print(f"⚠️ Aviso: El directorio '{dir_dataset}' no existe. Saltando...")
            continue
            
        # 2. Buscar todos los archivos .csv dentro de ese directorio
        patron_csv = os.path.join(dir_dataset, "**", "*.csv")
        archivos_csv = glob.glob(patron_csv, recursive=True)
        
        for ruta_csv in archivos_csv:
            total_archivos_revisados += 1
            nombre_archivo_corto = os.path.relpath(ruta_csv, data_path)
            
            try:
                # Leer el dataframe actual
                df = pd.read_csv(ruta_csv)
                
                # Comprobación de seguridad: verificar que las columnas necesarias existan
                columnas_requeridas = ["reorder", "operation", "avg_result_size"]
                if not all(col in df.columns for col in columnas_requeridas):
                    continue
                
                # --- VALIDACIÓN 1: neighborsPrune (Aplica a none, polar y cartesian) ---
                df_prune = df[df["operation"] == "neighborsPrune"]
                
                # Extraer medias asegurando que el modo exista en el subset de prune
                v_none_prune = df_prune[df_prune["reorder"] == "none"]["avg_result_size"].mean()
                v_polar_prune = df_prune[df_prune["reorder"] == "polar"]["avg_result_size"].mean()
                v_cartesian_prune = df_prune[df_prune["reorder"] == "cartesian"]["avg_result_size"].mean()
                
                # --- VALIDACIÓN 2: neighborsStruct (Solo aplica a none y polar) ---
                df_struct = df[df["operation"] == "neighborsStruct"]
                
                v_none_struct = df_struct[df_struct["reorder"] == "none"]["avg_result_size"].mean()
                v_polar_struct = df_struct[df_struct["reorder"] == "polar"]["avg_result_size"].mean()
                
                # Variables para controlar si encontramos errores en este archivo concreto
                error_detectado = False
                detalles_error = []

                # Comprobación de neighborsPrune (Si alguno es NaN/None, saltamos la comparación)
                if all(v is not None and not np.isnan(v) for v in [v_none_prune, v_polar_prune, v_cartesian_prune]):
                    err_polar_prune = not np.isclose(v_none_prune, v_polar_prune, rtol=1e-5, atol=1e-8)
                    err_cart_prune = not np.isclose(v_none_prune, v_cartesian_prune, rtol=1e-5, atol=1e-8)
                    
                    if err_polar_prune or err_cart_prune:
                        error_detectado = True
                        detalles_error.append(f"  -> [neighborsPrune]")
                        detalles_error.append(f"     Base 'none': {v_none_prune}")
                        detalles_error.append(f"     'polar':     {v_polar_prune}  {'⚠️ ¡FALLO!' if err_polar_prune else '✅ OK'}")
                        detalles_error.append(f"     'cartesian': {v_cartesian_prune}  {'⚠️ ¡FALLO!' if err_cart_prune else '✅ OK'}")

                # Comprobación de neighborsStruct
                if all(v is not None and not np.isnan(v) for v in [v_none_struct, v_polar_struct]):
                    err_polar_struct = not np.isclose(v_none_struct, v_polar_struct, rtol=1e-5, atol=1e-8)
                    
                    if err_polar_struct:
                        error_detectado = True
                        detalles_error.append(f"  -> [neighborsStruct]")
                        detalles_error.append(f"     Base 'none': {v_none_struct}")
                        detalles_error.append(f"     'polar':     {v_polar_struct}  ⚠️ ¡FALLO!")

                # Si se levantó alguna alerta, la reportamos
                if error_detectado:
                    total_errores += 1
                    print(f"❌ DISCREPANCIA DETECTADA en: {nombre_archivo_corto}")
                    for linea in detalles_error:
                        print(linea)
                    print("-" * 75)
                    
            except Exception as e:
                print(f"💥 Error al leer el archivo {nombre_archivo_corto}: {e}")

    # 6. Evaluación final
    print("-" * 75)
    print(f"Resumen: Revisados {total_archivos_revisados} archivos CSV en total.")
    
    if total_errores == 0:
        print("\n✅ TEST COMPLETADO CON ÉXITO")
        print("La integridad de vecinos es correcta para todas las combinaciones de 'reorder' y 'operation' en:")
        for dataset in clouds:
            print(f' - {dataset}')
        print()
    else:
        print(f"\n🚨 ERRORES: Se encontraron {total_errores} archivos con discrepancias de vecinos.")

In [3]:
verificar_integridad_vecinos(SUBSET_DATA, ALL_CLOUDS)

Iniciando test de validación de resultados estructurado por operaciones...
---------------------------------------------------------------------------
---------------------------------------------------------------------------
Resumen: Revisados 135 archivos CSV en total.

✅ TEST COMPLETADO CON ÉXITO
La integridad de vecinos es correcta para todas las combinaciones de 'reorder' y 'operation' en:
 - bildstein_station1_xyz_intensity_rgb
 - sg27_station8_intensity_rgb
 - 5085_54320
 - 5095_54440
 - Lille_0
 - Paris_Luxembourg_6
 - PNOA_2024_PNR_489-4672_NPC01
 - Mar18_train
 - Mar18_test

